<a href="https://colab.research.google.com/github/MK-ayaz/Unsloth_Studio_Colab/blob/main/Telegram_bot_that_can_scrape_videos_of_tiktok_users_and_sending_to_the_bot2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import sys
# Install libraries
!{sys.executable} -m pip install python-telegram-bot yt-dlp --quiet
print('Setup complete: python-telegram-bot and yt-dlp installed.')

Setup complete: python-telegram-bot and yt-dlp installed.


In [2]:
import yt_dlp
import os
import asyncio
from telegram import Update, InputMediaVideo
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
from google.colab import userdata

# Securely load the token
try:
    TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
except Exception:
    TELEGRAM_BOT_TOKEN = 'YOUR_BOT_TOKEN_HERE'

def get_tiktok_videos(url):
    # Using browser impersonation to bypass hard pagination limits
    ydl_opts = {
        'extract_flat': 'in_playlist',
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': True,
        'geo_bypass': True,
        'playlist_items': '1-10000',
        'impersonate': 'chrome', # Mimics browser fingerprinting
        'http_headers': {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        }
    }
    video_data = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=False)
            if info and 'entries' in info:
                for entry in info['entries']:
                    if entry and 'url' in entry:
                        desc = entry.get('description') or entry.get('title') or 'TikTok Video'
                        video_data.append({'url': entry['url'], 'title': desc})
            elif info and 'url' in info:
                desc = info.get('description') or info.get('title') or 'TikTok Video'
                video_data.append({'url': info['url'], 'title': desc})
        except Exception as e:
            print(f"Extraction error: {e}")
    return video_data

def download_single_video(url):
    ydl_opts = {
        'format': 'best',
        'outtmpl': 'tiktok_%(id)s.%(ext)s',
        'quiet': True,
        'no_warnings': True,
        'impersonate': 'chrome',
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.prepare_filename(info)

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    text = update.message.text
    if 'tiktok.com' in text:
        status_msg = await update.message.reply_text("🔍 *Deep Scraping Profile (Browser Impersonation Active)...*", parse_mode='Markdown')
        try:
            videos = await asyncio.to_thread(get_tiktok_videos, text)
            count = len(videos)
            if count == 0:
                await status_msg.edit_text("⚠️ No videos found. Ensure the profile is public.")
                return
            await status_msg.edit_text(f"📦 *Found {count} video(s)!*\nDownloading batches...", parse_mode='Markdown')
            batch_size = 10
            for i in range(0, count, batch_size):
                batch = videos[i:i + batch_size]
                media_group = []
                temp_files = []
                for j, v_info in enumerate(batch):
                    try:
                        url = v_info['url']
                        display_title = (v_info['title'][:150] + '...') if len(v_info['title']) > 150 else v_info['title']
                        file_path = await asyncio.to_thread(download_single_video, url)
                        if os.path.exists(file_path):
                            temp_files.append(file_path)
                            caption = f"🎥 {display_title}\n\n({i + j + 1}/{count})"
                            media_group.append(InputMediaVideo(open(file_path, 'rb'), caption=caption))
                    except Exception as e: print(f"Error processing {url}: {e}")
                if media_group:
                    try: await update.message.reply_media_group(media=media_group)
                    except Exception as e: await update.message.reply_text(f"❌ Batch Error: {e}")
                    finally:
                        for f in temp_files:
                            try: os.remove(f)
                            except: pass
                await asyncio.sleep(3)
            await update.message.reply_text("✅ *Done!*", parse_mode='Markdown')
        except Exception as e: await update.message.reply_text(f"❌ *Fatal Error:* `{e}`", parse_mode='Markdown')

bot_application = None

async def main():
    global bot_application
    if bot_application: await stop_bot()
    bot_application = Application.builder().token(TELEGRAM_BOT_TOKEN).build()
    bot_application.add_handler(CommandHandler('start', lambda u, c: u.message.reply_text('Bot Ready')))
    bot_application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    await bot_application.initialize()
    await bot_application.updater.start_polling()
    await bot_application.start()
    print('Bot is running with full browser impersonation...')

async def stop_bot():
    global bot_application
    if bot_application:
        try:
            if bot_application.updater.running: await bot_application.updater.stop()
            await bot_application.stop()
            await bot_application.shutdown()
        except Exception as e: print(f'Shutdown error: {e}')
        finally: bot_application = None
    tasks = [t for t in asyncio.all_tasks() if t is not asyncio.current_task()]
    for task in tasks: task.cancel()
    print('Bot stopped.')

In [2]:
import asyncio
# Starting the bot with updated video titling logic
try:
    await main()
except NameError:
    print('❌ Error: Please run the cell above (63cdd6be) to register the updated functions.')

Bot is running with deep pagination support...


In [3]:
# Execute the improved stopping function to fully halt the bot
await stop_bot()

CancelledError: 

In [5]:
import yt_dlp

# --- TEST CONFIGURATION ---
TEST_PROFILE_URL = 'https://www.tiktok.com/@chickennuggetsop' # Replace with a profile you know has > 350 videos

ydl_opts = {
    'extract_flat': 'in_playlist',
    'quiet': False,
    'no_warnings': False,
    'ignoreerrors': True,
    'playlist_items': '1-1000', # Try to get 1000 for this test
}

print(f'Starting diagnostic scrape for: {TEST_PROFILE_URL}')

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    try:
        info = ydl.extract_info(TEST_PROFILE_URL, download=False)
        if info and 'entries' in info:
            entries = list(info['entries'])
            total_found = len(entries)
            print(f'\n--- RESULTS ---')
            print(f'Total videos extracted: {total_found}')

            if total_found > 0:
                print(f'First video: {entries[0].get("url")}')
                print(f'Last video: {entries[-1].get("url")}')

            if total_found <= 350:
                print('\n⚠️ ALERT: Still stuck under the pagination limit. We may need to add a User-Agent or Cookies.')
        else:
            print('No entries found at all.')
    except Exception as e:
        print(f'Error during diagnostic: {e}')

Starting diagnostic scrape for: https://www.tiktok.com/@chickennuggetsop
[tiktok:user] Extracting URL: https://www.tiktok.com/@chickennuggetsop
[tiktok:user] chickennuggetsop: Downloading user webpage


[download] Downloading playlist: chickennuggetsop
[tiktok:user] chickennuggetsop: Downloading page 1
[tiktok:user] chickennuggetsop: Downloading page 2
[tiktok:user] chickennuggetsop: Downloading page 3
[tiktok:user] chickennuggetsop: Downloading page 4
[tiktok:user] chickennuggetsop: Downloading page 5
[tiktok:user] chickennuggetsop: Downloading page 6
[tiktok:user] chickennuggetsop: Downloading page 7
[tiktok:user] chickennuggetsop: Downloading page 8
[tiktok:user] chickennuggetsop: Downloading page 9
[tiktok:user] chickennuggetsop: Downloading page 10
[tiktok:user] chickennuggetsop: Downloading page 11
[tiktok:user] chickennuggetsop: Downloading page 12
[tiktok:user] chickennuggetsop: Downloading page 13
[tiktok:user] chickennuggetsop: Downloading page 14
[tiktok:user] chickennuggetsop: Downloading page 15
[tiktok:user] chickennuggetsop: Downloading page 16
[tiktok:user] chickennuggetsop: Downloading page 17
[tiktok:user] chickennuggetsop: Downloading page 18
[tiktok:user] chickennu